# LUBB AI — remote GPU server

Runs the meeting AI (Whisper speech-to-text + an Ollama LLM) on Colab's GPU so the hub on your
computer doesn't need one. The hub reaches it through a free Cloudflare tunnel.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. **Runtime → Run all**. The first run takes ~5–10 min (downloads the models).
3. Copy the lines printed by the last cell into `backend/.env` on your computer and restart the hub.

The tunnel URL and token change every time this notebook restarts. Colab disconnects idle or long
sessions, so start it shortly before a meeting.

In [ ]:
# Settings
LLM_MODEL = "qwen2.5:14b"         # T4 (15 GB): 14b fits next to Whisper; use "qwen2.5:7b" for faster minutes
WHISPER_MODEL = "large-v3-turbo"

import secrets
AI_TOKEN = secrets.token_urlsafe(24)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install Ollama, faster-whisper and cloudflared
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip -q install faster-whisper fastapi "uvicorn[standard]" httpx
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Start Ollama, download the LLM and keep it loaded in GPU memory
import json, subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull $LLM_MODEL
preload = json.dumps({"model": LLM_MODEL, "keep_alive": -1}).encode()
urllib.request.urlopen("http://127.0.0.1:11434/api/generate", data=preload, timeout=600).read()
print("LLM loaded")

In [ ]:
%%writefile server.py
"""LUBB AI remote GPU server (runs on Google Colab, started by lubb_ai_colab.ipynb).

One URL serves both models, so the hub needs a single tunnel:
  POST /transcribe   16 kHz mono 16-bit PCM body -> {"text", "language"}   (faster-whisper)
  POST /api/chat     forwarded to the local Ollama, streamed back           (LLM)
Every request must carry "Authorization: Bearer $AI_TOKEN".
"""

import os

import httpx
import numpy as np
from fastapi import Depends, FastAPI, HTTPException, Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import StreamingResponse
from faster_whisper import WhisperModel

TOKEN = os.environ["AI_TOKEN"]
OLLAMA = "http://127.0.0.1:11434"
whisper = WhisperModel(os.environ.get("WHISPER_MODEL", "large-v3-turbo"), device="cuda", compute_type="float16")


def auth(request: Request) -> None:
    if request.headers.get("authorization") != f"Bearer {TOKEN}":
        raise HTTPException(401, "Bad or missing AI_TOKEN")


app = FastAPI(title="LUBB AI GPU server", dependencies=[Depends(auth)])


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "whisper": os.environ.get("WHISPER_MODEL"), "llm": os.environ.get("LLM_MODEL")}


def _transcribe(pcm: bytes, language: str | None) -> dict:
    audio = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768.0
    segments, info = whisper.transcribe(audio, language=language, vad_filter=True)
    # segments is lazy: consume it here, inside the worker thread
    return {"text": " ".join(s.text.strip() for s in segments), "language": info.language}


@app.post("/transcribe")
async def transcribe(request: Request, language: str | None = None) -> dict:
    return await run_in_threadpool(_transcribe, await request.body(), language)


@app.post("/api/chat")
async def chat(request: Request) -> StreamingResponse:
    client = httpx.AsyncClient(timeout=None)
    upstream = await client.send(
        client.build_request(
            "POST", f"{OLLAMA}/api/chat", content=await request.body(), headers={"Content-Type": "application/json"}
        ),
        stream=True,
    )

    async def body():
        try:
            async for chunk in upstream.aiter_raw():
                yield chunk
        finally:
            await upstream.aclose()
            await client.aclose()

    return StreamingResponse(body(), status_code=upstream.status_code, media_type="application/x-ndjson")

In [ ]:
# Start the server (loads Whisper on the GPU) and wait until it answers
import glob, os, site
env = dict(os.environ, AI_TOKEN=AI_TOKEN, WHISPER_MODEL=WHISPER_MODEL, LLM_MODEL=LLM_MODEL)
# faster-whisper needs the CUDA/cuDNN libraries that ship with Colab's PyTorch
cuda_libs = glob.glob(os.path.join(site.getsitepackages()[0], "nvidia", "*", "lib"))
env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + [os.environ.get("LD_LIBRARY_PATH", "")])
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8001"],
    env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
req = urllib.request.Request("http://127.0.0.1:8001/health", headers={"Authorization": f"Bearer {AI_TOKEN}"})
for _ in range(120):
    try:
        print(urllib.request.urlopen(req).read().decode()); break
    except Exception:
        if server.poll() is not None:
            print(open("server.log").read()); raise RuntimeError("server crashed, see log above")
        time.sleep(5)
else:
    print(open("server.log").read()); raise RuntimeError("server did not start in 10 min")

In [ ]:
# Open a public HTTPS tunnel and print the hub settings
import re
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8001"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    if m := re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read()):
        url = m.group(0); break
    time.sleep(1)
if not url:
    print(open("tunnel.log").read()); raise RuntimeError("tunnel did not start")
time.sleep(5)  # give DNS a moment
print(f"""
Put this in backend/.env on your computer, then restart the hub:

STT_ENGINE=remote
STT_URL={url}
LLM_ENGINE=ollama
OLLAMA_URL={url}
LLM_MODEL={LLM_MODEL}
AI_TOKEN={AI_TOKEN}
""")

Keep this tab open during the meeting. If something fails, run the cell below to see the logs.

In [ ]:
!tail -n 30 server.log
!tail -n 10 ollama.log
!tail -n 10 tunnel.log